# Chương 5: Thực hành - Xác suất và Thống kê cho Học máy (Probability & Statistics for ML)

**Mục tiêu bài thực hành:**
1. **Mô phỏng Định lý Giới hạn Trung tâm (CLT):** Kiểm chứng sự hình thành tự nhiên của phân phối Gaussian từ việc lấy trung bình mẫu.
2. **Hình học hóa Phân phối Chuẩn 2D:** Khảo sát tác động của ma trận hiệp phương sai (đẳng hướng, đường chéo, có tương quan) lên elip đồng mức xác suất.
3. **Tự cài đặt Gaussian Naive Bayes Classifier:** Xây dựng bộ phân loại xác suất từ đầu (**from scratch**) bằng `NumPy` và trực quan hóa ranh giới quyết định.
4. **Đối kháng MLE vs. MAP:** Mô phỏng sự khác biệt giữa Ước lượng hợp lý cực đại (MLE) và Ước lượng cực đại hậu nghiệm (MAP) trên tập dữ liệu ít mẫu và nhiều mẫu.
5. **Lý thuyết thông tin thực nghiệm:** Tính toán và vẽ đồ thị Entropy Shannon, Cross-Entropy và Phân kỳ KL (Kullback-Leibler Divergence).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Cố định hạt giống ngẫu nhiên
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Mô phỏng Định lý Giới hạn Trung tâm (Central Limit Theorem - CLT)

Lấy mẫu từ phân phối đều $U(0, 1)$ (không hề có dạng hình chuông). Khi lấy trung bình của $N$ biến ngẫu nhiên độc lập, phân phối của giá trị trung bình sẽ hội tụ về phân phối chuẩn $\mathcal{N}(\mu, \sigma^2/N)$ khi $N$ tăng lên.

In [ ]:
n_samples = 10000
sample_sizes = [1, 2, 5, 30]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for idx, N in enumerate(sample_sizes):
    # Sinh ma trận n_samples x N từ phân phối đều Uniform(0, 1)
    samples = np.random.uniform(0, 1, size=(n_samples, N))
    # Tính giá trị trung bình mẫu dọc theo mỗi hàng
    sample_means = np.mean(samples, axis=1)
    
    ax = axes[idx]
    ax.hist(sample_means, bins=40, density=True, color='#3498db', edgecolor='white', alpha=0.8)
    ax.set_title(f"N = {N} biến cộng gộp", fontsize=11, fontweight='bold')
    ax.set_xlabel("Giá trị trung bình")
    ax.set_xlim(0, 1)
    
plt.suptitle("Minh họa Định lý Giới hạn Trung tâm (CLT) từ Phân phối Đều Uniform(0, 1)", fontsize=14, y=1.03)
plt.tight_layout()
plt.show()

## 2. Hình học hóa Phân phối Chuẩn 2 chiều (2D Multivariate Gaussian)

Khảo sát dạng hình học của các đường mức xác suất (Contours) ứng với 3 dạng ma trận hiệp phương sai $\boldsymbol{\Sigma}$:
1. **Đẳng hướng (Isotropic):** $\boldsymbol{\Sigma} = \sigma^2 \mathbf{I}$ (đường mức là hình tròn).
2. **Đường chéo (Diagonal):** $\text{Cov}(X_1, X_2) = 0, \sigma_1 \neq \sigma_2$ (elip song song trục tọa độ).
3. **Có tương quan (Full / Correlated):** $\text{Cov}(X_1, X_2) \neq 0$ (elip bị xoay nghiêng).

In [ ]:
def multivariate_gaussian_pdf(X, Y, mu, sigma):
    pos = np.dstack((X, Y))
    n = mu.shape[0]
    sigma_det = np.linalg.det(sigma)
    sigma_inv = np.linalg.inv(sigma)
    N = np.sqrt((2 * np.pi)**n * sigma_det)
    fac = np.einsum('...k,kl,...l->...', pos - mu, sigma_inv, pos - mu)
    return np.exp(-fac / 2) / N

x = np.linspace(-4, 4, 150)
y = np.linspace(-4, 4, 150)
X, Y = np.meshgrid(x, y)
mu = np.array([0.0, 0.0])

sigmas = [
    ("Đẳng hướng (Isotropic)\nCov = 0, Var1 = Var2", np.array([[1.5, 0.0], [0.0, 1.5]])),
    ("Đường chéo (Diagonal)\nCov = 0, Var1 > Var2", np.array([[2.5, 0.0], [0.0, 0.7]])),
    ("Có tương quan dương\nCov = 1.2 (Elip xoay nghiêng)", np.array([[2.0, 1.3], [1.3, 1.8]]))
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, (title, cov_matrix) in enumerate(sigmas):
    Z = multivariate_gaussian_pdf(X, Y, mu, cov_matrix)
    ax = axes[idx]
    contour = ax.contourf(X, Y, Z, levels=12, cmap='viridis')
    ax.contour(X, Y, Z, levels=6, colors='white', linewidths=0.7, alpha=0.6)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_aspect('equal')
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.6)
    ax.axvline(0, color='gray', linestyle='--', linewidth=0.6)

plt.suptitle("Hình học Đường mức Phân phối Chuẩn 2 Chiều (2D Gaussian Contours)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Cài đặt Bộ phân loại Gaussian Naive Bayes từ đầu (from scratch)

In [ ]:
class GaussianNaiveBayesScratch:
    def __init__(self):
        self.classes = None
        self.mean = {}
        self.var = {}
        self.priors = {}
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        
        for c in self.classes:
            X_c = X[y == c]
            self.mean[c] = np.mean(X_c, axis=0)
            # Bổ sung hằng số làm mịn eps để tránh phương sai bằng 0
            self.var[c] = np.var(X_c, axis=0) + 1e-9
            self.priors[c] = X_c.shape[0] / float(n_samples)
            
    def _calculate_likelihood(self, class_idx, x):
        mean = self.mean[class_idx]
        var = self.var[class_idx]
        # Tính hàm mật độ Gaussian từng chiều (giả định độc lập có điều kiện)
        numerator = np.exp(- (x - mean)**2 / (2 * var))
        denominator = np.sqrt(2 * np.pi * var)
        return np.sum(np.log(numerator / denominator))
        
    def predict(self, X):
        predictions = []
        for x in X:
            posteriors = []
            for c in self.classes:
                prior = np.log(self.priors[c])
                log_likelihood = self._calculate_likelihood(c, x)
                posterior = prior + log_likelihood
                posteriors.append(posterior)
            predictions.append(self.classes[np.argmax(posteriors)])
        return np.array(predictions)

# Sinh dữ liệu phân loại 2 cụm Gaussian
n_pts = 150
X_c0 = np.random.randn(n_pts, 2) * 1.2 + np.array([-1.5, -1.0])
X_c1 = np.random.randn(n_pts, 2) * 1.0 + np.array([2.0, 1.8])
X_data = np.vstack([X_c0, X_c1])
y_data = np.array([0] * n_pts + [1] * n_pts)

gnb = GaussianNaiveBayesScratch()
gnb.fit(X_data, y_data)
preds = gnb.predict(X_data)
acc = np.mean(preds == y_data)
print(f"[Gaussian Naive Bayes from scratch] Độ chính xác: {acc * 100:.2f}%")

## 4. Đối kháng Ước lượng Tham số: MLE vs. MAP (Bernoulli Coin Toss)

Giả sử ta tung một đồng xu có xác suất ngửa thực tế $\theta^* = 0.7$.
- **MLE:** $\hat{\theta}_{\text{MLE}} = \frac{k}{N}$ (dễ bị chệch cực đoan khi mẫu nhỏ $N=3$).
- **MAP:** Đặt tiên nghiệm $\text{Beta}(\alpha=5, \beta=5)$ thể hiện niềm tin rằng đồng xu thường cân bằng ($0.5$).
  $$\hat{\theta}_{\text{MAP}} = \frac{k + \alpha - 1}{N + \alpha + \beta - 2}$$

In [ ]:
true_p = 0.7
alpha_prior = 5
beta_prior = 5

sample_counts = np.arange(1, 120)
mle_estimates = []
map_estimates = []

# Sinh chuỗi các lần tung ngẫu nhiên theo true_p
flips = np.random.binomial(1, true_p, size=len(sample_counts))
k_heads = np.cumsum(flips)

for N, k in zip(sample_counts, k_heads):
    mle = k / N
    map_est = (k + alpha_prior - 1) / (N + alpha_prior + beta_prior - 2)
    mle_estimates.append(mle)
    map_estimates.append(map_est)

plt.figure(figsize=(12, 5))
plt.axhline(true_p, color='crimson', linestyle='--', linewidth=2, label=f'Giá trị thực (True p = {true_p})')
plt.plot(sample_counts, mle_estimates, label='MLE (Dễ bất ổn khi N nhỏ)', color='#e67e22', linewidth=1.8)
plt.plot(sample_counts, map_estimates, label='MAP (Ổn định nhờ Tiên nghiệm)', color='#2980b9', linewidth=2.2)

plt.title("So sánh Ước lượng Tham số: MLE vs MAP theo Kích thước Dữ liệu N", fontsize=13, fontweight='bold')
plt.xlabel("Số lần tung (N)", fontsize=11)
plt.ylabel("Xác suất ước lượng (theta)", fontsize=11)
plt.ylim(0, 1.1)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 5. Lý thuyết Thông tin: Entropy và Phân kỳ KL

In [ ]:
# 1. Đồ thị Entropy nhị phân H(p)
p_vals = np.linspace(1e-5, 1 - 1e-5, 200)
entropy_vals = - (p_vals * np.log2(p_vals) + (1 - p_vals) * np.log2(1 - p_vals))

# 2. Tính phân kỳ KL giữa phân phối thực P và mô hình Q
def kl_divergence(p, q):
    eps = 1e-12
    p = np.clip(p, eps, 1.0)
    q = np.clip(q, eps, 1.0)
    return np.sum(p * np.log(p / q))

P_real = np.array([0.7, 0.2, 0.1])      # Nhãn thực tế (3 lớp)
Q_good = np.array([0.65, 0.25, 0.10])   # Mô hình dự đoán tốt
Q_bad = np.array([0.15, 0.50, 0.35])    # Mô hình dự đoán kém

kl_good = kl_divergence(P_real, Q_good)
kl_bad = kl_divergence(P_real, Q_bad)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(p_vals, entropy_vals, color='#8e44ad', linewidth=2.5)
ax.scatter(0.5, 1.0, color='red', s=80, zorder=5, label='Cực đại tại p = 0.5 (1.0 bit)')
ax.set_title("Hàm Entropy Shannon Nhị phân H(p)", fontsize=12, fontweight='bold')
ax.set_xlabel("Xác suất thành công p", fontsize=11)
ax.set_ylabel("Entropy (bits)", fontsize=11)
ax.legend()
plt.tight_layout()
plt.show()

print("=== PHÂN KỲ KULLBACK-LEIBLER (KL DIVERGENCE) ===")
print(f"Phân phối thực tế P      : {P_real}")
print(f"Mô hình tốt Q_good       : {Q_good}  -> D_KL(P || Q) = {kl_good:.4f}")
print(f"Mô hình kém Q_bad        : {Q_bad}  -> D_KL(P || Q) = {kl_bad:.4f}")

## 6. Tổng kết bài học thực hành

1. **Định lý Giới hạn Trung tâm:** Giải thích vì sao phân phối Gaussian là giả định phổ biến nhất cho nhiễu trong tự nhiên và học máy.
2. **Phân phối Chuẩn 2D:** Ma trận hiệp phương sai định hình trực tiếp ranh giới quyết định hình học trong không gian đặc trưng.
3. **MLE vs. MAP:** Khi dữ liệu ít, MLE có nguy cơ quá khớp cao; MAP đóng vai trò như một cơ chế chính quy hóa (Regularization) thông qua việc áp đặt phân phối tiên nghiệm để ghìm giữ tham số.
4. **Lý thuyết thông tin:** Tối thiểu hóa Cross-Entropy Loss hay Negative Log-Likelihood chính là việc kéo phân kỳ KL giữa dự đoán mô hình và phân phối dữ liệu thực về mức tối thiểu.